# Compact transition tables (paper-ready)
This notebook generates **tiny, side-by-side** transition tables:
- **Global → Object** and **Global → Background** on the **same row**
- One page per **k** (aggregated over eps/models unless you change grouping)
- Exports a **multi-page PDF** suitable for papers

**Assumption:** you already have a `df` DataFrame in memory that includes:
- `image, model, k, eps, semantic`
- `status_short` (short labels like `CE (imm)`, `Unk (B&B)`, etc.)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Order (keeps categories even if empty)
ORDER = [
    "CE (imm)", "CE (init)", "Safe (init)", "Safe (B&B)",
    "Unk (B&B)", "Unk (no B&B)", "Other"
]


In [ ]:
def transition_crosstab(df, a="Global", b="Object", order=ORDER, value_col="status_short"):
    """Return crosstab with TOTAL row/col; ALWAYS includes full ORDER even if absent."""
    pivot = df.pivot_table(
        index=["image", "model", "k", "eps"],
        columns="semantic",
        values=value_col,
        aggfunc="first"
    )
    if a not in pivot.columns or b not in pivot.columns:
        return None
    t = pivot[[a, b]].dropna()
    tab = pd.crosstab(t[a], t[b], dropna=False)

    rows = list(order) + [x for x in tab.index if x not in order]
    cols = list(order) + [x for x in tab.columns if x not in order]
    tab = tab.reindex(index=rows, columns=cols, fill_value=0)

    tab.loc["TOTAL"] = tab.sum(axis=0)
    tab["TOTAL"] = tab.sum(axis=1)
    return tab


In [ ]:
def render_table_to_ax(
    ax,
    tab: pd.DataFrame,
    title: str,
    fontsize: int = 6,
    xscale: float = 0.88,
    yscale: float = 0.88,
    diag_color: str = "#fff6b3",
    offdiag_color: str = "#dbeafe",
    total_color: str = "#f1f5f9",
):
    """Paper-tiny Matplotlib table renderer."""
    ax.axis("off")
    data = tab.values.astype(int)
    row_labels = tab.index.tolist()
    col_labels = tab.columns.tolist()

    tbl = ax.table(
        cellText=data,
        rowLabels=row_labels,
        colLabels=col_labels,
        cellLoc="center",
        rowLoc="center",
        loc="center",
    )

    tbl.auto_set_font_size(False)
    tbl.set_fontsize(fontsize)
    tbl.scale(xscale, yscale)

    nrows, ncols = tab.shape
    for (r, c), cell in tbl.get_celld().items():
        # Column headers (r==0) and row header labels (c==-1)
        if r == 0 or c == -1:
            cell.set_text_props(weight="bold")
            cell.set_facecolor("white")
            cell.set_edgecolor("#cbd5e1")
            cell.set_linewidth(0.6)
            continue

        rr = r - 1
        cc = c
        rlab = row_labels[rr]
        clab = col_labels[cc]
        val = int(tab.iloc[rr, cc])

        cell.set_edgecolor("#cbd5e1")
        cell.set_linewidth(0.5)

        if rlab == "TOTAL" or clab == "TOTAL":
            cell.set_facecolor(total_color)
            cell.set_text_props(weight="bold")
        elif rlab == clab:
            cell.set_facecolor(diag_color)
            cell.set_text_props(weight="bold")
        elif val != 0:
            cell.set_facecolor(offdiag_color)
            cell.set_text_props(weight="bold")
        else:
            cell.set_facecolor("white")

    ax.set_title(title, fontsize=8, fontweight="bold", loc="left", pad=4)


In [ ]:
def export_pair_tables_by_k(
    df: pd.DataFrame,
    pdf_path: str = "transition_tables_by_k_compact.pdf",
    groupby_cols=("k",),
    title_prefix: str = "",
    figsize=(7.0, 2.1),
    fontsize: int = 6,
    xscale: float = 0.88,
    yscale: float = 0.88,
):
    """One PDF page per group; each page has Global→Object and Global→Background side-by-side."""
    with PdfPages(pdf_path) as pdf:
        for key, dfg in df.groupby(list(groupby_cols), dropna=False):
            if not isinstance(key, tuple):
                key = (key,)
            key_str = ", ".join([f"{c}={v}" for c, v in zip(groupby_cols, key)])
            page_title = (title_prefix + key_str).strip()

            tab_go = transition_crosstab(dfg, "Global", "Object", order=ORDER, value_col="status_short")
            tab_gb = transition_crosstab(dfg, "Global", "Background", order=ORDER, value_col="status_short")

            if tab_go is None and tab_gb is None:
                continue

            fig = plt.figure(figsize=figsize)
            ax1 = fig.add_axes([0.02, 0.08, 0.47, 0.84])
            ax2 = fig.add_axes([0.51, 0.08, 0.47, 0.84])

            if tab_go is not None:
                render_table_to_ax(ax1, tab_go, "Global → Object", fontsize=fontsize, xscale=xscale, yscale=yscale)
            else:
                ax1.axis("off")
                ax1.set_title("Global → Object (missing)", fontsize=8, fontweight="bold", loc="left", pad=4)

            if tab_gb is not None:
                render_table_to_ax(ax2, tab_gb, "Global → Background", fontsize=fontsize, xscale=xscale, yscale=yscale)
            else:
                ax2.axis("off")
                ax2.set_title("Global → Background (missing)", fontsize=8, fontweight="bold", loc="left", pad=4)

            if page_title:
                fig.text(0.02, 0.98, page_title, ha="left", va="top", fontsize=9, fontweight="bold")

            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

    return pdf_path


In [ ]:
# ---- RUN THIS ----
# One page per k (aggregated over eps/models)
pdf_path = export_pair_tables_by_k(
    df,
    pdf_path="transition_tables_by_k_compact.pdf",
    groupby_cols=("k",),
    figsize=(7.0, 2.1),
    fontsize=6,
    xscale=0.86,
    yscale=0.86,
)
print("Saved:", pdf_path)

# If you want per (k, eps) instead (more pages), uncomment:
# pdf_path = export_pair_tables_by_k(df, "transition_tables_by_k_eps_compact.pdf", groupby_cols=("k","eps"), figsize=(7.0, 2.1), fontsize=6, xscale=0.86, yscale=0.86)
# print("Saved:", pdf_path)
